In [ ]:
from sentence_transformers import SentenceTransformer


model = SentenceTransformer('all-MiniLM-L6-v2')

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
    )

In [ ]:
query = "How do run Docker on Windows"
query_vector = model.encode(query)
course='mlops-zoomcamp'

results = vs_index.search(query_vector,
num_results=5,
filter_dict={'course':course}
)

In [4]:
results

[{'id': '1bc7beaf5d',
  'course': 'mlops-zoomcamp',
  'section': 'Module 1: Introduction',
  'question': 'Install docker in WSL2 without installing Docker Desktop',
  'answer': 'If you want to install Docker in WSL2 on Windows without Docker Desktop, follow these steps:\n\n1. **Install Docker**\n\n   You can ignore the warnings during installation.\n   \n   ```bash\n   curl -fsSL https://get.docker.com -o get-docker.sh\n   sudo sh get-docker.sh\n   ```\n   \n2. **Add Your User to the Docker Group**\n   \n   ```bash\n   sudo usermod -aG docker $USER\n   ```\n\n3. **Enable the Docker Service**\n   \n   ```bash\n   sudo systemctl enable docker.service\n   ```\n\n4. **Test the Installation**\n\n   Verify that both Docker and Docker Compose are installed successfully.\n   \n   ```bash\n   docker --version\n   docker compose version\n   docker run hello-world\n   ```\n\n5. **Ensure Docker Starts Automatically**\n   \n   If the service does not start automatically after restarting WSL, update

In [6]:
from rag_helper import RAGBase
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self,query,num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course':self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# Load .env file
load_dotenv()

# Read API key
api_key = os.getenv("OPENAI_API_KEY")

# Initialize client
client = OpenAI(api_key=api_key)

# # Example request
# response = client.chat.completions.create(
#     model="gpt-4.1-mini",
#     messages=[
#         {"role": "user", "content": "Hello!"}
#     ]
# )

# print(response.choices[0].message.content)

In [14]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

vector_assistant = RAGVector(
    instructions=instructions,
    embedder=model,
    index=vs_index,
    llm_client=client,
    course=course
)

In [15]:
print(vector_assistant.rag(query))

To run Docker on Windows, use WSL2.

If you want to install Docker in WSL2 on Windows without Docker Desktop:

1. Install Docker:
```bash
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
```

2. Add your user to the Docker group:
```bash
sudo usermod -aG docker $USER
```

3. Enable the Docker service:
```bash
sudo systemctl enable docker.service
```

4. Test it:
```bash
docker --version
docker compose version
docker run hello-world
```

5. If Docker does not start automatically after restarting WSL, add the startup snippet to your `.profile` or `.zprofile`.

If you are setting up WSL on Windows, you can also install WSL with:
```bash
wsl --install
```
